In [ ]:
import sys 
import analysis_utils
sys.path.append("../")

from checkpoint import CheckPoint

### Incorrect Rule Application Analysis (Table 3)

In [ ]:
# load model:
rand_perm20 = CheckPoint.from_pt(path="../models/batching_experiments/MLC_batchrand_dallstudy1_nep20.pt")
model = rand_perm20.load_model()
test = rand_perm20.load_dataloaders("../data/deprecated/all_transformations_study1", use_datasets=["test"], batch_size=1_000, verbose=False)[0]

Loading model that has completed 22 of 20 epochs
	batch size: 32
	number of steps: 115,740
	best val loss achieved: 0.9758
MLC specs:
	1,400,606 parameters
	3 encoder layers
	3 decoder layers
	8 attention heads
	128 embedding size
	512 feedforward layer sizes
	gelu activation function
	p=0.1 dropout


In [ ]:
# obtain predictions of the model for the entire dataset:
predictions = analysis_utils.predict_dataset(test, model)
# filter for successor and predecessor rules only:
predictions = predictions[predictions["transformation"].isin(["pred", "succ"])]

def get_outcomes(x):
    """convert the entries in the column `applied transformation`  (computed in `analysis_utils.predict_dataset()`) to outcome labels"""
    if "correct" in x:
        return "correct"
    elif x == "none/other":
        return "other incorrect"
    else:
        return "applied " + x
    
predictions["outcome"] = predictions["applied_transformation"].apply(lambda x: get_outcomes(x))
error_analysis_pred_succ = predictions.groupby("transformation")["outcome"].value_counts(normalize=True)

error_analysis_pred_succ = error_analysis_pred_succ * 100
error_analysis_pred_succ

transformation  outcome     
pred            applied succ    76.424051
                correct         14.873418
                incorrect        8.702532
succ            applied pred    85.093168
                correct         14.906832
Name: proportion, dtype: float64